# Using the ZMART-controller

The controller offers a short, fixed list of commands, and every microscope answers them in the same form. A workflow written against these commands runs on any microscope that has a ZMART driver. This notebook goes through each command once, on the mock driver, a simulated microscope that is included with the controller and requires no hardware. Run the cells in order.

## Connecting

`get_instruments` lists the drivers installed on this computer, with the connection each one uses. The mock driver is always listed. `connect` opens the connection; from then on, every command is called as `mic.<command>`. The second argument is optional and is passed to the driver unchanged; here it tells the mock driver where to save its images.

In [ ]:
from zmart_controller import mic

mic.get_instruments()

In [ ]:
mic.connect("mock", {"output_root": "images"})

## The microscope description

Every command answers with two fields: `success`, whether the driver carried out the request, and `content`, the information it returns. The content of `get_info` is a description of the microscope in plain language: what each setting means, its unit and bounds, and which objectives are fitted. It contains what the other commands cannot report, and is worth reading once for every new microscope.

In [ ]:
print(mic.get_info()["content"]["description"])

## Position

Positions are given in micrometres from the origin, a point chosen and saved once for this microscope, so that a position refers to the same place on the sample in every session. Each axis has one or more motors; on the mock driver, z has a coarse motor and a piezo for fine steps. The `canvas` of an axis is the range in which a picture can show anything: the travel of the stage plus half a field of view. Positions should be planned inside it.

In [ ]:
mic.get_actuators()["content"]

In [ ]:
mic.get_xyz()["content"]

## Moving

`set_xyz` takes all three axes, so a script always states the full target position. When the answer is returned, the stage has arrived. A move outside the travel is not carried out: the answer is `success: False`, and `content` gives the reason. A workflow should check `success` before continuing.

In [ ]:
mic.set_xyz(100, 50, 0)["content"]

In [ ]:
mic.set_xyz(99999, 0, 0)

## Settings

The state is a snapshot of the instrument settings. Its `changeable` part is what `set_state` can apply: on the mock driver, laser power, gain, exposure and the objective. Some or all of them can be sent; the others remain unchanged. The driver reads each setting back to confirm that it was applied, and the answer lists what was applied. A state is a plain dictionary, so it can be saved with `json` and applied again in a later session.

In [ ]:
mic.get_state()["content"]["changeable"]

In [ ]:
mic.set_state({"changeable": {"exposure_ms": 20.0}})["content"]

## Acquisition

Acquisition settings are the choices about how an image is captured and saved, as opposed to the instrument settings: the file format, or the planes of a z-stack. For each one, `options` lists the permitted values and `active` the value used when none is given.

`acquire` captures an image at the current position with the current settings and saves it, in one step. The label names this position in the saved files. The answer lists every saved file under `files`, and under `planes` records for every image plane its file, channel, depth and time point, and the stage position at which it was taken. These paths should be used as returned; where the files are stored is decided by the driver.

In [ ]:
mic.get_acquisition_settings()["content"]

In [ ]:
answer = mic.acquire(position_label="A1")
answer["content"]["files"]

A z-stack is a single acquisition with several planes. The mock driver starts at the current height and moves upwards; every plane records the height at which it was taken.

In [ ]:
answer = mic.acquire(position_label="A2", acquisition_settings={"z_planes": 3, "z_step_um": 2.0})
[(plane["z"], plane["z_um"]) for plane in answer["content"]["planes"]]

## Procedures

Procedures are routines offered by the microscope, such as autofocus. Each driver offers its own; the description states what a routine does and which options it accepts. A procedure is run by name, with its options given alongside the name.

In [ ]:
mic.get_procedures()["content"]

In [ ]:
mic.run_procedure({"name": "autofocus", "range_um": 10, "step_um": 1})["content"]

## Disconnecting

`disconnect` closes the connection. To use the microscope again, call `connect` again.

In [ ]:
mic.disconnect()

---

MIT license. Thom de Hoog, Center for Microscopy and Image Analysis (ZMB), University of Zurich. thom.dehoog@zmb.uzh.ch, thomdehoog@gmail.com.